In [ ]:
!pip install -U langchain-openai langchain-community langgraph faiss-cpu openai duckdb pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 k

# Memento: Cuando el LLM No Sabe lo que Tú Sabes

## El Problema

Imagina que le pides a un analista nuevo que calcule el valor del **Inventario Muerto** de tu empresa.

El analista es brillante. Sabe SQL, conoce bases de datos, puede hacer joins complejos. Pero no sabe qué significa "Inventario Muerto" *en tu empresa*. ¿Es producto sin stock? ¿Producto descontinuado? ¿Producto inactivo que aún ocupa bodega?

Ese es exactamente el problema que enfrenta un LLM cuando le damos consultas con jerga de dominio.

---

## Lo Que Vamos a Explorar

No es un problema de razonamiento. Es un problema de **conocimiento**.

Y la solución no es un prompt más largo — es **memoria**.

Hoy vamos a construir cuatro versiones del mismo sistema, cada una más sofisticada que la anterior:

| Fase | Arquitectura | Memoria | ¿Funciona? |
|------|-------------|---------|-------------|
| **A** | Agente único (ReAct) | Ninguna | No |
| **B** | Planificador + Ejecutor | Ninguna | No |
| **C** | Memento Simple | FAISS + humano | Sí, pero frágil |
| **D** | True Memento | Q-Learning + evaluador automático | Sí, y mejora solo |

---

## Nuestra Base de Datos

Trabajamos con un e-commerce pequeño en DuckDB (en memoria):

- **users** — 3 clientes (Ana, Carlos, Belén)
- **categories** — Electrónica, Ropa, Hogar
- **products** — 6 productos con precios, stock y estado activo/inactivo
- **orders** — 5 órdenes con montos y estatus

### Las Reglas Ocultas

Estas reglas **no están en la base de datos**. Viven en la cabeza del equipo de negocio:

- **Inventario Muerto**: Productos donde `stock_quantity > 0 AND is_active = FALSE`
- **Cliente VIP**: Usuarios cuya suma de `total_amount` en órdenes supera los $2,000

La pregunta central de esta clase es: **¿Cómo logra un agente LLM aprender estas reglas sin que las codifiquemos directamente en el prompt?**

---

## Herramientas Compartidas

Todos los agentes que construyamos hoy comparten dos herramientas:

1. **`search_schema`** — Busca esquemas de tablas en un índice FAISS (retrieval semántico sobre metadatos de la BD)
2. **`execute_sql`** — Ejecuta consultas SQL directamente en DuckDB

El problema nunca es la ejecución. El problema es saber *qué* ejecutar.

In [ ]:
import duckdb
import os
import getpass
import time
import torch
import torch.nn as nn
import torch.optim as optim
from typing import TypedDict, List
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END

# ==========================================
# 1. SETUP & API
# ==========================================
os.environ["OPENAI_API_KEY"] = getpass.getpass("Introduce tu OpenAI API Key: ")

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)
embeddings = OpenAIEmbeddings()

# ==========================================
# 2. DATABASE
# ==========================================
con = duckdb.connect(database=':memory:')
setup_sql = """
-- USERS
CREATE TABLE users (id INT PRIMARY KEY, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100) UNIQUE, created_at DATE);
INSERT INTO users VALUES
(1, 'Ana', 'Gomez', 'ana@example.com', '2023-01-15'),
(2, 'Carlos', 'Ruiz', 'carlos@example.com', '2023-03-20'),
(3, 'Belen', 'Soto', 'belen@example.com', '2023-05-10');

-- CATEGORIES
CREATE TABLE categories (id INT PRIMARY KEY, name VARCHAR(100));
INSERT INTO categories VALUES (1, 'Electronics'), (2, 'Apparel'), (3, 'Home');

-- PRODUCTS
CREATE TABLE products (id INT PRIMARY KEY, category_id INT, name VARCHAR(255), price DECIMAL(10, 2), stock_quantity INT, is_active BOOLEAN);
INSERT INTO products VALUES
(1, 1, 'Pro Laptop 15', 1299.99, 15, TRUE),
(2, 1, 'Wireless Mouse', 49.99, 50, TRUE),
(3, 2, 'Winter Coat', 199.99, 10, FALSE), -- Inventario Muerto
(4, 2, 'Summer Dress', 39.99, 0, FALSE),  -- No es inventario muerto (stock 0)
(5, 3, 'Table Lamp', 35.50, 20, FALSE),   -- Inventario Muerto
(6, 3, 'Desk Chair', 150.00, 5, TRUE);

-- ORDERS
CREATE TABLE orders (id INT PRIMARY KEY, user_id INT, total_amount DECIMAL(10, 2), status VARCHAR(20));
INSERT INTO orders VALUES
(101, 1, 1349.98, 'DELIVERED'),
(102, 2, 39.99, 'DELIVERED'),
(103, 1, 150.00, 'DELIVERED'),
(104, 3, 2500.00, 'DELIVERED'), -- Belen es VIP (>2000)
(105, 3, 100.00, 'SHIPPED');
"""
con.execute(setup_sql)

# Vector DB for Schemas
table_schemas_docs = [
    Document(page_content="Tabla: categories\nEsquema: id INT, name VARCHAR", metadata={"table_name": "categories"}),
    Document(page_content="Tabla: users\nEsquema: id INT, first_name VARCHAR, last_name VARCHAR, email VARCHAR, created_at DATE", metadata={"table_name": "users"}),
    Document(page_content="Tabla: products\nEsquema: id INT, category_id INT, name VARCHAR, price DECIMAL, stock_quantity INT, is_active BOOLEAN", metadata={"table_name": "products"}),
    Document(page_content="Tabla: orders\nEsquema: id INT, user_id INT, total_amount DECIMAL, status VARCHAR", metadata={"table_name": "orders"}),
]
schema_db = FAISS.from_documents(table_schemas_docs, embeddings)
schema_retriever = schema_db.as_retriever(search_kwargs={"k": 3})

# ==========================================
# 3. GLOBAL TOOLS
# ==========================================
@tool
def search_schema(query: str) -> str:
    """Busca esquemas de tablas SQL relevantes."""
    docs = schema_retriever.invoke(query)
    return "\n\n".join([f"Tabla: {d.metadata['table_name']}\n{d.page_content}" for d in docs])

@tool
def execute_sql(query: str) -> str:
    """Ejecuta una consulta SQL en DuckDB."""
    try:
        result_df = con.execute(query).df()
        return result_df.to_markdown(index=False) if not result_df.empty else "0 filas devueltas."
    except Exception as e:
        return f"Error SQL: {e}"

# Shared executor tool
base_executor = create_react_agent(model=model, tools=[search_schema, execute_sql])

Introduce tu OpenAI API Key: ··········


/tmp/ipykernel_3826/3498241258.py:91: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  base_executor = create_react_agent(model=model, tools=[search_schema, execute_sql])


# Fases A y B: Las Formas Elegantes de Fallar

## Fase A — El Agente Único

La arquitectura más simple posible: un solo agente ReAct con acceso a dos herramientas.

### ¿Qué hace?

1. Recibe la consulta del usuario
2. Decide qué herramienta usar (buscar esquema o ejecutar SQL)
3. Itera hasta tener una respuesta

### ¿Por qué falla?

Cuando le pedimos *"Calcula el valor del Inventario Muerto"*, el agente hace algo razonable: busca el esquema, ve las columnas, y escribe SQL. Pero **inventa** la definición de "Inventario Muerto" porque no tiene otra opción.

Podría asumir que es `stock_quantity = 0`, o `is_active = FALSE` sin considerar stock, o cualquier otra combinación. El LLM no distingue entre *no saber* y *adivinar con confianza*.

### Lección

> Un agente con herramientas potentes pero sin conocimiento de dominio es un programador junior con acceso a producción: peligroso no por lo que no puede hacer, sino por lo que hace sin entender.

---

## Fase B — Planificador + Ejecutor

Separamos la tarea en dos roles con un grafo de LangGraph:

```
[Planificador] → [Ejecutor] → FIN
```

### ¿Qué cambia?

- El **Planificador** genera un plan paso a paso (sin herramientas, solo razonamiento)
- El **Ejecutor** sigue ese plan usando las herramientas SQL

### ¿Por qué sigue fallando?

La descomposición es correcta. El plan tiene pasos lógicos, bien numerados, estructurados. Pero el Planificador **tampoco sabe** qué es un "Cliente VIP".

La diferencia con la Fase A es sutil: en vez de alucinación directa en el SQL, ahora la alucinación está en el plan. El ejecutor sigue fielmente un plan incorrecto.

### Lección

> Descomponer una tarea en subtareas no crea conocimiento nuevo. Si ningún nodo del sistema tiene la información correcta, la arquitectura no importa — solo distribuyes el error de forma más elegante.

---

## Pregunta Para Discusión

¿Por qué el planificador inventa una definición en lugar de decir "no sé qué significa Inventario Muerto"?

Piensen en esto desde la perspectiva del entrenamiento del modelo:

- Los LLMs están optimizados para ser **útiles** y **completar tareas**
- Decir "no sé" rara vez aparece como la respuesta correcta en los datos de entrenamiento
- El modelo tiene suficiente contexto semántico ("muerto" + "inventario") para construir una definición plausible

Este es un problema fundamental de **calibración**: el modelo no distingue entre lo que sabe con certeza y lo que está interpolando.

---

## El Estado Hasta Aquí

Tenemos un sistema que razona bien y ejecuta bien, pero que **no sabe lo que no sabe**. Lo que necesitamos no es más razonamiento — es **memoria**.

In [ ]:
# ==========================================
# FASE A: EL AGENTE ÚNICO
# ==========================================
print("\n" + "="*60)
print("🔴 FASE A: EL AGENTE ÚNICO (Sin planificación, sin memoria)")
print("="*60)

system_prompt_single = "Eres un experto en SQL. Usa 'search_schema' para ver tablas y 'execute_sql' para correr consultas."
single_agent = create_react_agent(model=model, tools=[search_schema, execute_sql], prompt=system_prompt_single)

query_jargon_1 = "Calcula el valor de nuestro 'Inventario Muerto'."
print(f"🗣️ [Usuario]: {query_jargon_1}")

for event in single_agent.stream({"messages": [("user", query_jargon_1)]}, stream_mode="values"):
    if "messages" in event:
        event["messages"][-1].pretty_print()

print("\n❌ EXPLICACIÓN: El agente no entiende la regla de un inventario muerto.")
time.sleep(2)


🔴 FASE A: EL AGENTE ÚNICO (Sin planificación, sin memoria)
🗣️ [Usuario]: Calcula el valor de nuestro 'Inventario Muerto'.
================================ Human Message =================================

Calcula el valor de nuestro 'Inventario Muerto'.


/tmp/ipykernel_3826/2141746431.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  single_agent = create_react_agent(model=model, tools=[search_schema, execute_sql], prompt=system_prompt_single)


================================== Ai Message ==================================
Tool Calls:
  search_schema (call_03PrAS1zGFsYPgmsVSk1c4Bq)
 Call ID: call_03PrAS1zGFsYPgmsVSk1c4Bq
  Args:
    query: Inventario Muerto
================================= Tool Message =================================
Name: search_schema

Tabla: products
Tabla: products
Esquema: id INT, category_id INT, name VARCHAR, price DECIMAL, stock_quantity INT, is_active BOOLEAN

Tabla: categories
Tabla: categories
Esquema: id INT, name VARCHAR

Tabla: users
Tabla: users
Esquema: id INT, first_name VARCHAR, last_name VARCHAR, email VARCHAR, created_at DATE
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_HOsKYVeqsH2YlNw9Y6rSZOaD)
 Call ID: call_HOsKYVeqsH2YlNw9Y6rSZOaD
  Args:
    query: -- Definamos 'Inventario Muerto' como productos activos con stock pero sin ventas recientes
-- Primero, veamos las tablas disponibles para ventas
SELECT * FROM products

In [ ]:
# ==========================================
# FASE B: PLANIFICADOR + EJECUTOR
# ==========================================
print("\n" + "="*60)
print("🟡 FASE B: PLANIFICADOR + EJECUTOR (Sin memoria)")
print("="*60)

class BaseAgentState(TypedDict):
    user_query: str
    plan: str
    final_answer: str

def basic_executor_node(state: BaseAgentState):
    prompt = f"Resuelve: '{state['user_query']}'.\nSIGUE ESTE PLAN:\n{state['plan']}"
    result = base_executor.invoke({"messages": [HumanMessage(content=prompt)]})
    return {"final_answer": result["messages"][-1].content}

def basic_planner_node(state: BaseAgentState):
    prompt = "Eres el Planificador. Crea un plan paso a paso para que el ejecutor escriba SQL. No uses herramientas."
    response = model.invoke([SystemMessage(content=prompt), HumanMessage(content=state['user_query'])])
    return {"plan": response.content}

basic_workflow = StateGraph(BaseAgentState)
basic_workflow.add_node("planner", basic_planner_node)
basic_workflow.add_node("executor", basic_executor_node)
basic_workflow.set_entry_point("planner")
basic_workflow.add_edge("planner", "executor")
basic_workflow.add_edge("executor", END)
basic_app = basic_workflow.compile()

query_jargon_2 = "¿Quiénes son nuestros 'Clientes VIP' y cuánto han gastado en total?"
print(f"🗣️ [Usuario]: {query_jargon_2}")

res = basic_app.invoke({"user_query": query_jargon_2})
print(f"\n📝 PLAN GENERADO:\n{res['plan']}\n\n⚙️ RESPUESTA FINAL:\n{res['final_answer']}")

print("\n❌ EXPLICACIÓN: Divide bien la tarea, pero inventa la regla de los 'Clientes VIP'. Sigue fallando.")
time.sleep(2)


🟡 FASE B: PLANIFICADOR + EJECUTOR (Sin memoria)
🗣️ [Usuario]: ¿Quiénes son nuestros 'Clientes VIP' y cuánto han gastado en total?

📝 PLAN GENERADO:
Para responder a esta pregunta, necesitamos seguir un plan paso a paso para escribir la consulta SQL adecuada. Asumiré que tienes una base de datos con al menos dos tablas relevantes: una tabla de clientes y una tabla de ventas o transacciones.

### Plan paso a paso para escribir la consulta SQL:

1. **Identificar las tablas relevantes:**
   - Tabla de clientes (por ejemplo, `clientes`) que contiene información sobre los clientes, incluyendo un campo que indica si son 'VIP' o no.
   - Tabla de ventas o transacciones (por ejemplo, `ventas`) que contiene registros de las compras realizadas por los clientes, incluyendo el importe gastado y el identificador del cliente.

2. **Determinar el campo que identifica a los 'Clientes VIP':**
   - Puede ser un campo booleano o un campo de categoría en la tabla `clientes`, por ejemplo, `tipo_cliente = '

# Fase C: Memento Simple — Memoria Episódica con FAISS

## La Idea Central

Si un agente falla, y un humano le explica *por qué* falló, ¿puede el agente **recordar** esa corrección la próxima vez que enfrente un problema similar?

Eso es exactamente lo que hace la memoria episódica: almacena experiencias pasadas (tarea, plan, resultado, corrección) y las recupera cuando aparece una tarea parecida.

---

## El Archivista Simple

La clase `SimpleArchivist` es un almacén de casos en FAISS con dos operaciones:

**Guardar un caso** — Cada caso contiene:
- La tarea original
- El plan que se usó
- Si fue éxito o fracaso
- La corrección o regla proporcionada por el humano

**Recuperar casos** — Dado un nuevo query, busca por similitud semántica los casos más cercanos en el espacio de embeddings.

---

## El Flujo Completo

```
[Planificador con Memoria] → [Ejecutor] → FIN
                ↑
        (consulta FAISS por
         casos similares)
```

El planificador ahora recibe dos cosas: la consulta del usuario **y** los casos pasados más relevantes. Su prompt le indica que lea la corrección/regla del caso previo y la incorpore en el nuevo plan.

---

## La Demostración en Dos Actos

### Intento 1 — FAISS Vacío

- El usuario pregunta por el Inventario Muerto
- No hay casos previos → el planificador opera sin memoria
- Falla (igual que en Fase B)
- **Un humano interviene** y proporciona la regla correcta: *"Inventario Muerto = stock_quantity > 0 AND is_active = FALSE. Multiplica price × stock_quantity."*
- El caso se guarda en FAISS

### Intento 2 — FAISS con Memoria

- El usuario pregunta algo similar sobre Inventario Muerto
- FAISS recupera el caso anterior por similitud
- El planificador lee la corrección y genera un plan con los filtros SQL correctos
- El ejecutor produce la respuesta correcta

**Funciona.**

---

## ¿Qué Resuelve?

- El agente ahora puede **aprender de sus errores** (con ayuda humana)
- El conocimiento de dominio se acumula en un vector store consultable
- Cada corrección humana beneficia a todas las consultas futuras similares

## ¿Qué No Resuelve?

### 1. Dependencia del humano
Cada nueva regla de negocio requiere que alguien falle, que un humano lo note, y que ese humano escriba la corrección. No escala.

### 2. Recuperación ingenua
FAISS usa similitud coseno pura. Un caso puede ser semánticamente similar pero **inútil** (o peor, dañino). Ejemplo:

> Caso almacenado: Tarea sobre "Inventario Muerto" con un plan **incorrecto** que obtuvo reward 0.

La similitud coseno no distingue entre un caso exitoso y uno fallido sobre el mismo tema. Recupera ambos con el mismo score.

### 3. Sin degradación elegante
Con 10,000 casos en FAISS, la probabilidad de recuperar ruido aumenta. No hay mecanismo para *ponderar* la calidad de los casos — solo su proximidad semántica.

---

## Pregunta Para Discusión

Si la similitud semántica no es suficiente para rankear memorias, ¿qué señal adicional necesitamos?

Pista: necesitamos una señal que capture **qué tan útil fue un caso para resolver la tarea**, no solo qué tan parecido es al query actual. Eso es exactamente lo que un reward proporciona.

In [ ]:
# ==========================================
# FASE C: MEMENTO SIMPLE (FAISS + Humano)
# ==========================================
print("\n" + "="*60)
print("🔵 FASE C: MEMENTO SIMPLE (Memoria Episódica Básica + Feedback Humano)")
print("="*60)

class SimpleArchivist:
    def __init__(self):
        self.db = FAISS.from_texts(["init"], embeddings)
        self.db.delete([self.db.index_to_docstore_id[0]])
        self.count = 0

    def save_case(self, task: str, plan: str, success: bool, feedback: str):
        status = "SUCCESS" if success else "FAILURE"
        content = f"[RESULTADO: {status}]\nTarea: {task}\nPlan Usado: {plan}\nCorrección/Regla: {feedback}"
        doc = Document(page_content=content, metadata={"task": task})
        self.db.add_documents([doc])
        self.count += 1

    def retrieve(self, task: str, k: int = 1) -> str:
        if self.count == 0: return "No hay casos previos."
        docs = self.db.similarity_search(task, k=k)
        return "\n\n".join([d.page_content for d in docs])

simple_archivist = SimpleArchivist()

def simple_memento_planner_node(state: BaseAgentState):
    past_cases = simple_archivist.retrieve(state['user_query'])
    prompt = f"""Eres un Planificador con Memoria Simple. Crea un plan para el ejecutor de SQL.
    === CASOS PASADOS (FAISS Similarity) ===
    {past_cases}
    ===============================
    Instrucciones: Lee atentamente la 'Corrección/Regla' e inyecta esa lógica en tu nuevo plan."""
    response = model.invoke([SystemMessage(content=prompt), HumanMessage(content=state['user_query'])])
    return {"plan": response.content}

simple_memento_workflow = StateGraph(BaseAgentState)
simple_memento_workflow.add_node("planner", simple_memento_planner_node)
simple_memento_workflow.add_node("executor", basic_executor_node)
simple_memento_workflow.set_entry_point("planner")
simple_memento_workflow.add_edge("planner", "executor")
simple_memento_workflow.add_edge("executor", END)
simple_memento_app = simple_memento_workflow.compile()

def run_simple_memento(query: str, mock_success=None, mock_feedback=None):
    print(f"\n🗣️ [Usuario]: {query}")
    final_plan = ""
    for event in simple_memento_app.stream({"user_query": query}):
        if "planner" in event:
            final_plan = event["planner"]["plan"]
            print(f"📝 [Planificador Simple]: Plan generado y pasado al ejecutor.")
        elif "executor" in event:
            print(f"⚙️ [Ejecutor SQL]:\n{event['executor']['final_answer']}")

    if mock_feedback:
        print(f"\n👨‍💻 [Feedback Humano Simulado]: Éxito: {mock_success} | Regla: {mock_feedback}")
        simple_archivist.save_case(query, final_plan, mock_success, mock_feedback)
        print("💾 Memoria guardada en FAISS.")

print("\n--- INTENTO 1 (FAISS Vacío) ---")
run_simple_memento("Calcula el valor del Inventario Muerto.",
                   mock_success=False,
                   mock_feedback="El Inventario Muerto son productos donde stock_quantity > 0 AND is_active = FALSE. Multiplica price * stock_quantity.")

time.sleep(2)
print("\n--- INTENTO 2 (FAISS con Memoria) ---")
run_simple_memento("¿Cuánto valor tenemos en Inventario Muerto ahora mismo?")

print("\n⚠️ EXPLICACIÓN: Funciona, pero depende de retroalimentación humana constante y la búsqueda FAISS básica puede recuperar casos irrelevantes si la base de datos crece mucho.")
time.sleep(2)


🔵 FASE C: MEMENTO SIMPLE (Memoria Episódica Básica + Feedback Humano)

--- INTENTO 1 (FAISS Vacío) ---

🗣️ [Usuario]: Calcula el valor del Inventario Muerto.
📝 [Planificador Simple]: Plan generado y pasado al ejecutor.
⚙️ [Ejecutor SQL]:
En la base de datos disponible, la tabla que contiene información relevante para inventarios es la tabla "products". Esta tabla tiene las siguientes columnas: id, category_id, name, price, stock_quantity, is_active.

Para definir el Inventario Muerto, necesito saber qué criterio usar para identificar esos productos. Por ejemplo, podría ser productos con stock_quantity mayor a cero pero que estén inactivos (is_active = false), o productos sin ventas recientes (pero no tengo datos de ventas detallados aquí).

¿Podrías especificar cómo se define el Inventario Muerto en tu contexto? Por ejemplo, ¿productos inactivos con stock? ¿O algún otro criterio?

👨‍💻 [Feedback Humano Simulado]: Éxito: False | Regla: El Inventario Muerto son productos donde stock_quan

# Puente Conceptual: Recuperación de Memoria como un Problema de RL

## La Analogía

Recuperar la memoria correcta para una tarea es un **problema de decisión secuencial**. Podemos modelarlo como un MDP:

| Componente MDP | En Nuestro Sistema |
|---|---|
| **Estado** | El embedding de la consulta actual del usuario |
| **Acción** | Elegir qué caso pasado recuperar del banco de memoria |
| **Reward** | La evaluación de si la respuesta final fue correcta (0.0 o 1.0) |
| **Transición** | El caso recuperado influye en el plan → el plan influye en el SQL → el SQL produce un resultado evaluable |

---

## ¿Por Qué Q-Learning?

En Q-Learning clásico, aprendemos una función Q(s, a) que estima el **valor esperado** de tomar la acción *a* en el estado *s*.

En nuestro sistema:

- **Q(query, caso)** estima qué tan probable es que recuperar ese *caso* específico para ese *query* lleve a una respuesta correcta (reward = 1.0)
- Entrenamos esta función con experiencia real: cada vez que el sistema resuelve una tarea, observamos el reward y actualizamos Q

### La diferencia con FAISS puro

| FAISS (Fase C) | Q-Network (Fase D) |
|---|---|
| Rankea por **similitud semántica** | Rankea por **utilidad esperada** |
| Un caso fallido y uno exitoso sobre el mismo tema tienen el mismo score | El caso exitoso obtiene Q-value alto, el fallido Q-value bajo |
| No mejora con el tiempo | Mejora con cada interacción |
| Función fija (coseno) | Función aprendida (red neuronal) |

---

## La Arquitectura de la Red Q

```
Entrada: [embedding_query (1536) ‖ embedding_caso (1536)]
                          ↓
                   Linear(3072 → 512)
                          ↓
                        ReLU
                          ↓
                   Linear(512 → 1)
                          ↓
                       Sigmoid
                          ↓
                Salida: Q-value ∈ [0, 1]
```

La red recibe la **concatenación** de dos embeddings de OpenAI (1536 dims cada uno) y produce un escalar entre 0 y 1. La sigmoidea final nos da una interpretación natural: la probabilidad estimada de que ese caso lleve a éxito.

### ¿Por qué concatenación y no producto punto?

El producto punto captura similitud. La concatenación permite a la red aprender **interacciones no lineales** entre el query y el caso — puede aprender patrones como "para queries sobre inventario, los casos con feedback sobre filtros booleanos son más útiles que los casos sobre agregaciones".

---

## El Loop de Entrenamiento

```
1. Usuario hace un query
2. Q-Network rankea todos los casos del banco de memoria
3. Se recuperan los top-k casos
4. El planificador genera un plan usando esos casos
5. El ejecutor produce un resultado
6. El evaluador automático asigna un reward
7. Se hace un paso de gradiente: empujar Q(query, caso) hacia el reward observado
8. El caso se añade al banco de memoria
```

Esto es esencialmente **experience replay simplificado**: cada episodio genera un ejemplo de entrenamiento (query, caso, reward) que actualiza la red.

---

## Conexión con Otros Temas del Curso

Si ya vieron Q-Learning en otros contextos (precios dinámicos, market-making, enfriamiento de servidores), la intuición es idéntica. Lo que cambia es el espacio de acciones:

- **Pricing dinámico**: acción = qué precio poner
- **Market making**: acción = qué spread ofrecer
- **Memento**: acción = qué memoria recuperar

La función de valor siempre responde la misma pregunta: *"¿Qué tan bueno es tomar esta decisión en este contexto?"*

# Fase D: True Memento — Q-Learning + Evaluación Automática

## Los Tres Componentes Nuevos

La Fase D reemplaza dos cosas de la Fase C: el humano en el loop y la búsqueda por similitud pura. Lo hace con tres componentes que trabajan en conjunto.

---

## 1. El Archivista Paramétrico

`ParametricArchivist` reemplaza al `SimpleArchivist` de la Fase C. Mantiene un banco de casos como antes, pero la recuperación ya no es por similitud coseno — pasa por la red Q.

### Recuperación (`retrieve_top_k`)

Para cada caso en el banco de memoria:
1. Genera el embedding del query actual
2. Genera el embedding del caso candidato (tarea + plan)
3. Pasa ambos embeddings por la Q-Network
4. Obtiene un Q-value (0 a 1)
5. Ordena todos los casos por Q-value descendente
6. Devuelve los top-k

Esto significa que un caso semánticamente lejano pero históricamente útil puede rankearse arriba de un caso semánticamente cercano pero que llevó a fracasos.

### Entrenamiento (`save_and_train`)

Después de cada episodio:
1. El caso nuevo se añade al banco
2. Se genera el embedding del query y del caso
3. La red predice Q(query, caso)
4. Se compara contra el target binario: 1.0 si reward ≥ 0.5, sino 0.0
5. Se calcula BCE Loss y se hace un paso de gradiente

Un solo paso. Un solo ejemplo. Veremos por qué esto es una limitación importante.

---

## 2. El Planificador Memento

El planificador de la Fase D tiene un prompt sustancialmente diferente al de la Fase C.

### La Instrucción Vital

El prompt incluye esta directiva:

> *"El Ejecutor SQL NO tiene acceso a esta memoria. Por lo tanto, DEBES escribir las reglas matemáticas y filtros exactos en tu plan."*

¿Por qué esto importa? Porque en esta arquitectura, la memoria vive **solo en el planificador**. El ejecutor es un agente ReAct estándar que recibe un plan como texto. Si el plan dice "Paso 1: Filtra Clientes VIP", el ejecutor no sabe qué significa eso y va a alucinar la definición.

El planificador debe traducir el conocimiento recuperado en **instrucciones SQL literales**: "Paso 1: Agrupa la tabla orders por user_id y filtra donde SUM(total_amount) > 2000".

### Ejemplo Malo vs. Bueno

| ❌ Malo | ✅ Bueno |
|---|---|
| "Filtrar Inventario Muerto" | "Filtrar products WHERE stock_quantity > 0 AND is_active = FALSE" |
| "Identificar Clientes VIP" | "Agrupar orders por user_id, calcular SUM(total_amount), filtrar > 2000" |

Esta es una lección de diseño de prompts para sistemas multi-agente: **el conocimiento debe ser explícito en los puntos de transferencia entre agentes**.

---

## 3. El Evaluador Automático

Este es el componente que reemplaza al humano. Es un LLM (el mismo GPT-4.1-mini) con un prompt especializado que incluye:

### El Glosario de la Empresa

Un diccionario de reglas de negocio hardcodeado en el prompt del evaluador:
- "Inventario Muerto" = `stock_quantity > 0 AND is_active = FALSE`
- "Cliente VIP" = `SUM(total_amount) > 2000`

### Las Reglas de Evaluación

Tres reglas deliberadas:

1. **Asumir que los datos son correctos** — El evaluador no tiene acceso a la BD, así que no puede verificar números. Solo evalúa lógica.
2. **Evaluar solo la lógica del plan** — Si los pasos contienen los filtros correctos del glosario, el reward es 1.0.
3. **Criterios claros de fracaso** — Reward 0.0 si el plan asume reglas distintas a las del glosario.

### ¿Por qué funciona este diseño?

El evaluador es un **oráculo imperfecto pero consistente**. No necesita ejecutar SQL — solo necesita verificar que las condiciones lógicas en el plan coincidan con las del glosario. Es más barato, más rápido, y suficientemente confiable para generar señal de reward.

---

## El Grafo Completo

```
[Planificador] → [Ejecutor] → [Evaluador] → FIN
      ↑                              |
      |                              ↓
  Q-Network ← ← ← ← ← ← ←  save_and_train
  (recupera                    (actualiza pesos
   top-k casos)                con reward)
```

Tres nodos en el StateGraph de LangGraph: `planner → executor → evaluator`. Después del evaluador, el reward se usa para entrenar la Q-Network y el caso se almacena para futuras recuperaciones.

---

## La Secuencia de Prueba

### Intento 1 — Inventario Muerto (sin memoria)
- Banco vacío → planificador sin contexto previo
- El evaluador tiene el glosario → puede evaluar correctamente
- Si falla, se guarda con reward 0.0 y el feedback correcto

### Intento 2 — Inventario Muerto (con memoria)
- La Q-Network recupera el caso del intento 1
- Si el intento 1 tuvo reward 0.0, el feedback contiene la regla correcta
- El planificador inyecta la regla literal en el plan
- El evaluador da reward 1.0
- La Q-Network se actualiza: este caso sí fue útil → Q-value sube

### Intento 3 — Clientes VIP (nueva jerga)
- El banco tiene casos de inventario pero ninguno de VIP
- La Q-Network decide si esos casos son relevantes (probablemente no mucho)
- El evaluador evalúa contra la regla de VIP del glosario

### Intento 4 — Clientes VIP (con memoria de intento 3)
- Ahora hay un caso de VIP en el banco
- La Q-Network lo recupera con Q-value más alto
- El sistema aplica la regla correcta

**Observa los Q-values en cada intento.** Deberían mostrar cómo la red aprende a distinguir casos útiles de casos irrelevantes.

In [ ]:
# ============================================================
# 🟢 FASE D: TRUE MEMENTO (Q-Learning + Subtasks + Evaluación Automática)
# ============================================================
print("\n" + "="*60)
print("🟢 FASE D: TRUE MEMENTO (Q-Learning + Subtasks + Evaluación Automática)")
print("="*60)

class QNetwork(nn.Module):
    def __init__(self, embed_dim=1536):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim * 2, 512)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, state_emb, case_emb):
        x = torch.cat([state_emb, case_emb], dim=-1)
        x = self.relu(self.fc1(x))
        return self.sigmoid(self.fc2(x))

class ParametricArchivist:
    def __init__(self, embedder):
        self.embedder = embedder
        self.q_net = QNetwork()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=0.01)
        self.criterion = nn.BCELoss()
        self.case_bank = []

    def embed(self, text: str) -> torch.Tensor:
        vector = self.embedder.embed_query(text)
        return torch.tensor(vector, dtype=torch.float32)

    def retrieve_top_k(self, current_query: str, k: int = 2) -> str:
        if not self.case_bank: return "No hay casos previos en la memoria."
        state_emb = self.embed(current_query)
        scored_cases = []

        self.q_net.eval()
        with torch.no_grad():
            for case in self.case_bank:
                case_emb = self.embed(f"Task: {case['query']}\nPlan: {case['plan']}")
                q_value = self.q_net(state_emb, case_emb).item()
                scored_cases.append((q_value, case))

        scored_cases.sort(key=lambda x: x[0], reverse=True)
        top_k = scored_cases[:k]

        print("\n🔍 [Motor de Recuperación Q-Learning]:")
        for score, c in top_k:
            print(f"   -> Q-Value Estimado: {score:.4f} | Tarea: '{c['query']}'")

        return "\n\n".join([f"[Q-Score: {s:.4f} | Reward Pasado: {c['reward']}]\nTarea: {c['query']}\nPlan: {c['plan']}\nFEEDBACK: {c.get('feedback', 'N/A')}" for s, c in top_k])

    def save_and_train(self, query: str, plan: str, reward: float, feedback: str):
        self.case_bank.append({'query': query, 'plan': plan, 'reward': reward, 'feedback': feedback})
        self.q_net.train()
        self.optimizer.zero_grad()

        state_emb = self.embed(query)
        case_emb = self.embed(f"Task: {query}\nPlan: {plan}")

        q_pred = self.q_net(state_emb, case_emb).squeeze()
        target_val = 1.0 if reward >= 0.5 else 0.0
        target = torch.tensor(target_val, dtype=torch.float32)

        loss = self.criterion(q_pred, target)
        loss.backward()
        self.optimizer.step()
        print(f"🧠 [Red Q Actualizada] Predicción Inicial: {q_pred.item():.4f} | Pérdida (Loss): {loss.item():.4f}")

parametric_archivist = ParametricArchivist(embeddings)

class MementoState(TypedDict):
    user_query: str; case_memory: str; subtasks: List[str]; final_answer: str; reward: float

def true_planner_node(state: MementoState):
    past_cases = parametric_archivist.retrieve_top_k(state['user_query'])

    # PROMPT MEJORADO: Forzamos al planificador a traducir el feedback en reglas SQL literales.
    prompt = f"""Eres el Planificador Memento. Descompón la tarea en pasos.
    === CASOS PREVIOS (Q-Valued) ===
    {past_cases}
    ================================
    INSTRUCCIÓN VITAL:
    1. Revisa el FEEDBACK de los casos previos.
    2. El Ejecutor SQL NO tiene acceso a esta memoria. Por lo tanto, DEBES escribir las reglas matemáticas y filtros exactos en tu plan.
    Ejemplo Malo: "Paso 1: Filtrar Clientes VIP".
    Ejemplo Bueno: "Paso 1: Filtrar la tabla orders agrupando por usuario donde SUM(total_amount) > 2000".

    Devuelve UNA LISTA NUMERADA de pasos explícitos."""

    response = model.invoke([SystemMessage(content=prompt), HumanMessage(content=state['user_query'])])
    return {"subtasks": [step for step in response.content.split('\n') if step.strip()], "case_memory": past_cases}

def memento_executor_node(state: MementoState):
    plan_str = "\n".join(state['subtasks'])
    result = base_executor.invoke({"messages": [HumanMessage(content=f"Resuelve: '{state['user_query']}'.\nSIGUE ESTE PLAN ESTRICTAMENTE:\n{plan_str}")]})
    return {"final_answer": result["messages"][-1].content}

def automated_evaluator_node(state: MementoState):
    # PROMPT CORREGIDO: Forzamos al evaluador a no verificar los datos, sino la lógica.
    prompt = f"""Eres un Evaluador Automático Estricto.
    Pregunta Original: {state['user_query']}
    Respuesta Generada: {state['final_answer']}
    Pasos Usados: {state['subtasks']}

    GLOSARIO ESTRICTO DE LA EMPRESA:
    - "Inventario Muerto": Productos donde stock_quantity > 0 AND is_active = FALSE.
    - "Cliente VIP": Usuarios cuya suma de total_amount en orders es > 2000.

    REGLAS DE EVALUACIÓN VITALES (LEE ATENTAMENTE):
    1. ASUME QUE LOS DATOS SON 100% CORRECTOS: No tienes acceso a la base de datos. Confía en que los nombres (ej. Belén, Ana, Winter Coat) y los números devueltos en la 'Respuesta Generada' son reales. NO castigues a la respuesta por los datos.
    2. EVALÚA SOLO LA LÓGICA: Tu único trabajo es revisar los 'Pasos Usados'. Si los pasos incluyen los filtros matemáticos y lógicos exactos del glosario (ej. "stock_quantity > 0 AND is_active = FALSE" o "> 2000"), el REWARD DEBE SER 1.0.
    3. MOTIVOS PARA FALLAR (REWARD 0.0): Si los pasos asumen reglas distintas (ej. asumir que muerto es stock = 0), si la respuesta pide aclaraciones al usuario, o si no aplicó el glosario.

    Responde EXACTAMENTE en dos líneas:
    REWARD: [0.0 o 1.0]
    FEEDBACK: [Explicación concisa y directa citando la regla del glosario. Si es 1.0, felicita la lógica usada en los pasos.]"""

    texto = model.invoke([HumanMessage(content=prompt)]).content.strip()
    reward, feedback = 0.0, "Sin feedback"

    for linea in texto.split('\n'):
        if linea.startswith("REWARD:"):
            try: reward = float(linea.replace("REWARD:", "").strip())
            except: pass
        elif linea.startswith("FEEDBACK:"): feedback = linea.replace("FEEDBACK:", "").strip()

    print(f"⚖️ [Auto-Evaluador] Reward: {reward} | Feedback: {feedback}")
    parametric_archivist.save_and_train(state['user_query'], "\n".join(state['subtasks']), reward, feedback)
    return {"reward": reward}

m_mdp_workflow = StateGraph(MementoState)
m_mdp_workflow.add_node("planner", true_planner_node)
m_mdp_workflow.add_node("executor", memento_executor_node)
m_mdp_workflow.add_node("evaluator", automated_evaluator_node)
m_mdp_workflow.set_entry_point("planner")
m_mdp_workflow.add_edge("planner", "executor")
m_mdp_workflow.add_edge("executor", "evaluator")
m_mdp_workflow.add_edge("evaluator", END)
true_memento_app = m_mdp_workflow.compile()

def run_true_memento(query: str):
    print(f"\n🗣️ [Usuario]: {query}")
    for event in true_memento_app.stream({"user_query": query}):
        if "planner" in event:
            print("\n📝 [Planificador] Plan Generado:")
            for step in event["planner"]["subtasks"]: print(f"   {step}")
        if "executor" in event:
            print(f"\n⚙️ [Ejecutor SQL] Resultado Final:\n{event['executor']['final_answer']}")

# ==========================================
# SECUENCIA DE PRUEBA (4 ITERACIONES)
# ==========================================

print("\n--- INTENTO 1 Inventario Muerto ---")
run_true_memento("Calcula el valor financiero del Inventario Muerto.")

time.sleep(2)
print("\n--- INTENTO 2 Inventario Muerto ---")
run_true_memento("Haz una lista con los nombres de los productos que son Inventario Muerto.")

time.sleep(2)
print("\n--- INTENTO 3 Clientes VIP ---")
run_true_memento("¿Quiénes son nuestros Clientes VIP y cuánto gastaron?")

time.sleep(2)
print("\n--- INTENTO 4 Clientes VIP ---")
run_true_memento("Dime los correos de nuestros Clientes VIP.")


🟢 FASE D: TRUE MEMENTO (Q-Learning + Subtasks + Evaluación Automática)

--- INTENTO 1 Inventario Muerto ---

🗣️ [Usuario]: Calcula el valor financiero del Inventario Muerto.

📝 [Planificador] Plan Generado:
   Para calcular el valor financiero del Inventario Muerto, sigue estos pasos detallados:
   1. Identificar los productos en inventario que califican como "Inventario Muerto". Esto generalmente significa productos que no han tenido movimientos (ventas o uso) en un período determinado, por ejemplo, 12 meses.  
      - Consulta la tabla de inventario y la tabla de movimientos o ventas.  
      - Filtra los productos cuyo último movimiento fue hace más de 12 meses o que no han tenido movimientos en ese período.
   2. Obtener la cantidad actual en inventario de esos productos identificados como Inventario Muerto.  
      - Sumar la cantidad disponible en inventario para cada producto muerto.
   3. Obtener el costo unitario de cada producto muerto.  
      - Consultar la tabla de produc

# Discusión de Cierre: Limitaciones, Extensiones y Aplicaciones

## Lo Que Construimos

Un sistema que:
- Descompone tareas en subtareas (Planificador)
- Ejecuta SQL con herramientas (Ejecutor)
- Evalúa sus propios resultados (Evaluador automático)
- Almacena experiencia y aprende a recuperarla inteligentemente (Q-Network)

Pasamos de un agente que alucina reglas de negocio a uno que aprende a aplicarlas correctamente a través de experiencia.

---

## Limitaciones Honestas

### 1. El evaluador ya sabe las respuestas

El glosario de la empresa está hardcodeado en el prompt del evaluador. El sistema no *descubre* que "Inventario Muerto" significa `stock > 0 AND is_active = FALSE` — el evaluador ya lo sabe y el planificador aprende a alinearse con esa definición.

**Pregunta incómoda:** ¿Es circular?

**Respuesta matizada:** En producción, esto es útil cuando el glosario representa un estándar existente (reglas de compliance, definiciones del data dictionary, métricas oficiales del negocio). El valor no está en descubrir las reglas sino en enseñar al agente a *aplicarlas consistentemente* sin intervención humana cada vez.

### 2. Entrenamiento con batch size 1

Cada episodio produce un solo ejemplo de entrenamiento y hace un solo paso de gradiente. Esto genera:

- **Alta varianza** en las actualizaciones
- **Olvido catastrófico** potencial (un solo paso puede borrar lo aprendido)
- **Ineficiencia muestral** (cada experiencia se usa exactamente una vez)

Una implementación seria usaría un **experience replay buffer**: almacenar los últimos N ejemplos (query, caso, reward) y samplear mini-batches aleatorios para cada actualización. Esto estabiliza el entrenamiento y reutiliza experiencia pasada.

### 3. Sin target network

En Q-Learning estable (DQN), se usa una red target congelada para generar los targets de entrenamiento. Aquí el target es binario (0.0 o 1.0) así que el problema es menor, pero en una versión con rewards continuos sería importante.

### 4. Escalabilidad de la recuperación

`retrieve_top_k` itera sobre **todos** los casos del banco y pasa cada uno por la Q-Network. Con 10,000 casos esto se vuelve lento. Opciones:

- Prefiltrar con FAISS (top-50 por similitud) y luego rankear con la Q-Network (top-k por utilidad)
- Usar la Q-Network como un re-ranker sobre candidatos del retriever base

### 5. Reward binario

El evaluador solo devuelve 0.0 o 1.0. Pierde matices: un plan que aplica la regla correcta pero con un JOIN innecesario obtiene el mismo reward que un plan perfecto. Un reward más granular permitiría optimizar no solo corrección sino eficiencia.

---

## Extensiones Posibles (Ideas para Proyectos)

### A. Conflicto de feedback
¿Qué pasa si dos evaluaciones contradictorias se almacenan para tareas similares? Implementar un mecanismo de **incertidumbre** que detecte conflictos y los escale a revisión humana.

### B. Memoria jerárquica
Separar la memoria en niveles: **reglas de negocio** (alta permanencia, como el glosario), **patrones SQL** (media permanencia, como JOINs frecuentes), y **contexto de sesión** (baja permanencia, como filtros temporales). Cada nivel con su propia estrategia de recuperación.

### C. Evaluador aprendido
Reemplazar el evaluador basado en prompts por un modelo entrenado con feedback humano real (RLHF-style). Esto eliminaría la necesidad del glosario hardcodeado.

### D. Multi-paso con re-planificación
Si el evaluador da reward 0.0, en lugar de terminar, **re-planificar** con el feedback del evaluador inyectado en el prompt. Convertir el grafo lineal en un ciclo con máximo de reintentos.

---

## ¿Dónde Se Aplica Esto en la Industria?

El patrón "agente + memoria episódica + evaluación automática" aparece en contextos donde:

1. **El dominio tiene jerga especializada** que el LLM base no conoce
2. **Las reglas cambian** con el tiempo y hardcodearlas en prompts no escala
3. **El costo de error es significativo** (financiero, regulatorio, reputacional)

Ejemplos concretos:

- **Alertas de portafolio** — "Posición concentrada" significa cosas distintas en cada institución
- **Compliance regulatorio** — Reglas de la CNBV o Banxico que se actualizan periódicamente
- **Atención al cliente** — Políticas internas de devolución, garantías, escalamiento
- **Reporting financiero** — Métricas como EBITDA ajustado tienen definiciones específicas por empresa

---

## Resumen del Arco Completo

| | Conocimiento | Retrieval | Feedback | Mejora |
|---|---|---|---|---|
| **Fase A** | Ninguno | — | — | No |
| **Fase B** | Ninguno | — | — | No |
| **Fase C** | Episódico | Similitud coseno | Humano | Sí, manual |
| **Fase D** | Episódico | Q-Learning paramétrico | Evaluador LLM | Sí, automática |

La progresión no es solo técnica — es conceptual: pasamos de *"el modelo sabe todo"* a *"el modelo aprende de su experiencia"*.